# 1 Loading the data
I will load the raw data from the csv file

In [259]:
import pandas as pd
import numpy as np
from global_land_mask import globe
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.errors import EmptyDataError, ParserError

try:
    df=pd.read_csv("../data/raw/food-delivery.csv")
    print("file loaded successfully !")
except FileNotFoundError :
    print("Error: the specified file does not exists")
except EmptyDataError :
    print("Error: the file exists but contains no data")
except ParserError :
    print("Error: could not parse the file , might contains corrupted rows")
except UnicodeDecodeError :
    print("Error: Encoding issue !")
except Exception as e :
    print(f"Error: {e}")

file loaded successfully !


# 2 Inspecting the dataset
- check how many rows and columns are there
- the name of each column
- the type of each column
- inspecting the first and the last rows of the dataset
- how many missing values are there

In [260]:
# pandas.DataFrame.shape returns a tuple containing the dataframe's dimensions
shape = df.shape
print(f"there're {shape[0]} rows and {shape[1]} columns")

print("\n")
print("======================================================")
print("\n")

# pandas.DataFrame.columns returns a pandas.index object containing column labels
for column in df.columns :
    print(column)

print("\n")
print("======================================================")
print("\n")

# pandas.DataFrame.dtypes returns a series with the data type of each column
print(df.dtypes)

print("\n")
print("======================================================")
print("\n")

# pandas.DataFrame.head(x) returns the first x rows
print(df.head(5))
print("\t\t\t...........................")
# pandas.DataFrame.tail(x) returns the last x rows
print(df.tail(5))

print("\n")
print("======================================================")
print("\n")

# pandas.DataFrame.isna() return a boolean same-sized object that indicates if the values are Na it returns True for the missing values and False for valid values
# since Bool is a subclass of int True and False are considered Simultaneously 1 and 0
# and by using pandas.DataFrame.isna().sum() we get the sum of the True values that will work as a missing value count per column
print(df.isna().sum())
print("------------------------------------------------------")
# but if the missing values are masked as strings such as "NaN" or "word NaN" it will be considered as valid string values
# we will use pandas.Dataframe.replace() to replace them with true missing values
df = df.replace(r"NaN",np.nan,regex=True)
print(df.isna().sum())

print("\n")
print("======================================================")
print("\n")

there're 45593 rows and 20 columns




ID
Delivery_person_ID
Delivery_person_Age
Delivery_person_Ratings
Restaurant_latitude
Restaurant_longitude
Delivery_location_latitude
Delivery_location_longitude
Order_Date
Time_Orderd
Time_Order_picked
Weatherconditions
Road_traffic_density
Vehicle_condition
Type_of_order
Type_of_vehicle
multiple_deliveries
Festival
City
Time_taken(min)




ID                                 str
Delivery_person_ID                 str
Delivery_person_Age                str
Delivery_person_Ratings            str
Restaurant_latitude            float64
Restaurant_longitude           float64
Delivery_location_latitude     float64
Delivery_location_longitude    float64
Order_Date                         str
Time_Orderd                        str
Time_Order_picked                  str
Weatherconditions                  str
Road_traffic_density               str
Vehicle_condition                int64
Type_of_order                      str
Type_of_vehicle                 

# 3 Handling Problematic Columns
we need to convert the type of certain columns to an appropriate ones

In [261]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"],format="%d-%m-%Y",errors="coerce")
df["Time_Orderd"] = pd.to_datetime(df["Time_Orderd"],format="%H:%M:%S",errors="coerce")
df["Time_Order_picked"] = pd.to_datetime(df["Time_Order_picked"],format="%H:%M:%S",errors="coerce")

df = df.astype({
    "Delivery_person_Age":"Int64",
    "Delivery_person_Ratings":"Float64",
    "multiple_deliveries":"Int64"
})

df["Road_traffic_density"] = df["Road_traffic_density"].str.strip()

df["Time_taken(min)"] = df["Time_taken(min)"].str.replace("(min) ","")
df["Time_taken(min)"] = df["Time_taken(min)"].astype("Int64")
print(df.dtypes)

ID                                        str
Delivery_person_ID                        str
Delivery_person_Age                     Int64
Delivery_person_Ratings               Float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[us]
Time_Orderd                    datetime64[us]
Time_Order_picked              datetime64[us]
Weatherconditions                         str
Road_traffic_density                      str
Vehicle_condition                       int64
Type_of_order                             str
Type_of_vehicle                           str
multiple_deliveries                     Int64
Festival                                  str
City                                      str
Time_taken(min)                         Int64
dtype: object


# 4 Clening Parasitic Text
in certain columns their values might look like this "word value" the word is considered a parasitic text that we need to clean

In [262]:
df["Weatherconditions"] = df["Weatherconditions"].str.replace("conditions ","")
print(df["Weatherconditions"].head(5))
print(df["Weatherconditions"].tail(5))

0         Sunny
1        Stormy
2    Sandstorms
3         Sunny
4        Cloudy
Name: Weatherconditions, dtype: str
45588     Windy
45589     Windy
45590    Cloudy
45591    Cloudy
45592       Fog
Name: Weatherconditions, dtype: str


# 5 Handeling Missing Values
handling the missing data by filling it with an appropriate value

In [263]:
df["Delivery_person_Age"] = df["Delivery_person_Age"].fillna(int(df["Delivery_person_Age"].mean()))

df["Delivery_person_Ratings"] = df["Delivery_person_Ratings"].fillna(df["Delivery_person_Ratings"].median())

temp = df[df["Time_Orderd"].notna()]
difference = ((temp["Time_Order_picked"] - temp["Time_Orderd"]).dt.total_seconds()/60)%1440
average_minute_difference = int(difference.mean())
df["Time_Orderd"] = df["Time_Orderd"].fillna(df["Time_Order_picked"] - pd.to_timedelta(average_minute_difference,unit="m"))

df["Weatherconditions"] = df["Weatherconditions"].fillna(df["Weatherconditions"].mode()[0])


df = df.dropna(subset="Road_traffic_density")

df["multiple_deliveries"] = df["multiple_deliveries"].fillna(int(df["multiple_deliveries"].mean()))

df["Festival"] = df["Festival"].fillna(df["Festival"].mode()[0])

df["City"] = df["City"].fillna(df["City"].mode()[0])

print(df.isna().sum())


ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64


# 6 Detecting Invalid Values

In [264]:
for row in df.itertuples() :
    lat1 = row.Restaurant_latitude
    lon1 = row.Restaurant_longitude
    lat2 = row.Delivery_location_latitude
    lon2 = row.Delivery_location_longitude

    if globe.is_ocean(lat1,lon1):
        index = df[df["ID"] == row.ID].index
        df.drop(index=index,inplace=True)

    if globe.is_ocean(lat2,lon2):
        index = df[df["ID"] == row.ID].index
        df.drop(index=index,inplace=True)

cols = ["Restaurant_latitude","Restaurant_longitude"]
df[cols] = df[cols].abs()

# 7 Storing the Clean Data

In [265]:
df["Time_Orderd"] = df["Time_Orderd"].dt.time
df["Time_Order_picked"] = df["Time_Order_picked"].dt.time

df.to_csv("../data/processed/clean_data.csv",index=False)